# 🎮 Projet 07 : Mission "Architecte Data & Base de Données IoT"

Bienvenue dans ton septième projet pratique ! Tu incarnes l'ingénieur chargé de concevoir l'infrastructure de stockage et d'analyse de données d'un réseau électrique intelligent (*Smart Grid*).

---

## 💡 Le Concept du Stockage SQL pour l'Énergie

Dans un système IoT industriel, nous devons sauvegarder de manière permanente et sécurisée des millions de mesures envoyées par les capteurs :

* **Le Capteur IoT (Sensor) = La Source d'Information** : Il génère en continu des flux de données de télémétrie (tension en V, courant en A, puissance en W, énergie en kWh).
* **Le Code Python (Moteur SQL) = L'Acheminement & Traitement** : Il reçoit les trames de données, les valide et exécute les commandes SQL pour échanger avec le stockage.
* **La Base de Données SQL = Le Coffre-Fort Persistant** : Elle conserve les données de manière structurée sur disque, permettant des recherches ultra-rapides sans saturer la mémoire RAM.

![Architecture du stockage SQL IoT](images/architecture_sql_iot.png)

```text
[ Capteurs IoT ] ───(Télémétrie brute)───> [ Code Python (sqlite3) ] ───(Requêtes SQL)───> [ Base SQLite (.db) ]

## 🛠️ Configuration de l'environnement : Python, SQLite & `uv`

Avant de démarrer le code, faisons un point rapide sur l'installation des outils.

---

### 💡 Bonne nouvelle : SQLite est déjà inclus dans Python !

Contrairement à PostgreSQL, MySQL ou MongoDB, **SQLite ne nécessite aucune installation de package externe ni de serveur à configurer**. 

* **Bibliothèque standard** : Le module `sqlite3` est directement fourni avec Python.
* **Aucun `uv add sqlite` nécessaire** : Tu peux directement exécuter `import sqlite3` dans n'importe quel script Python.


### 💡 Modélisation Orientée Objet vs Modélisation Relationnelle (2 Tables)
En ingénierie IoT, on sépare la définition matérielle d'un capteur (statique) des flux de mesures qu'il produit (dynamique temporel).

## 🟢 Niveau 01 : De la POO à la Table SQL (Connexion & Structure)

Prendre le temps de bien assimiler la passerelle entre la POO (Programmation Orientée Objet) et le SQL est la meilleure approche. En POO, une classe est un plan de construction pour instancier des objets en mémoire RAM. En SQL, une table est ce même plan de construction, mais pensé pour structurer des lignes de données sauvegardées de façon permanente sur disque.

---

### 💡 Correspondance : Classe Python vs Table SQL

Voici une classe Python simple représentant une mesure de capteur et son équivalent exact sous forme de Table SQL :

```python
# En POO (Python) : Un objet vit temporairement dans la mémoire RAM
class ReleveEnergie:

    def __init__(
        self,
        sensor_id: str,
        timestamp: str,
        puissance_w: float,
        tension_v: float,
    ):
        self.sensor_id = sensor_id
        self.timestamp = timestamp
        self.puissance_w = puissance_w
        self.tension_v = tension_v


# Exemple d'instance
mesure_01 = ReleveEnergie("PANNEAU_SUD", "2026-08-30 14:00:00", 1250.5, 230.2)

| Attribut POO (Python) | Type Python | Colonne SQL | Type SQL | Rôle & Règle Métier |
| :--- | :--- | :--- | :--- | :--- |
| *(Aucun)* | `None` | `id` | `INTEGER` | Identifiant unique généré automatiquement (`PRIMARY KEY AUTOINCREMENT`). |
| `self.sensor_id` | `str` | `sensor_id` | `TEXT` | Nom ou code du capteur (`NOT NULL` = Obligatoire). |
| `self.timestamp` | `str` / `datetime` | `timestamp` | `DATETIME` | Date et heure précises du relevé (`NOT NULL`). |
| `self.puissance_w` | `float` | `puissance_w` | `REAL` | Puissance en Watts (`REAL` = Nombre à virgule). |
| `self.tension_v` | `float` | `tension_v` | `REAL` | Tension du réseau en Volts (`REAL`). |

---

### 🔍 Les Petits Détails qui Changent Tout en SQL

* **Pourquoi ajouter la colonne `id` en SQL ?** En Python, deux objets en mémoire sont distingués par leur adresse mémoire. En base de données, deux relevés peuvent avoir des valeurs identiques ; la colonne `id` sert donc de plaque d'immatriculation unique à chaque ligne.
* **`REAL` au lieu de `float` :** Chaque langage a son vocabulaire. Dans la norme SQL (et SQLite), un nombre à virgule s'appelle un type `REAL` (Nombre Réel).
* **`TEXT` au lieu de `str` :** De même, les chaînes de caractères `str` deviennent des types `TEXT`.
* **La contrainte `NOT NULL` :** C'est un gardien de sécurité. Si un capteur subit un bug et envoie une mesure sans préciser son nom (`sensor_id`), la base de données stoppe l'écriture pour éviter d'enregistrer une donnée corrompue.

### 📐 Diagramme de Classe UML (POO)

```text
┌────────────────────────────────────────┐
│             ReleveEnergie              │
├────────────────────────────────────────┤
│ + id : Optional[int]                   │
│ + sensor_id : str                      │
│ + timestamp : str / datetime           │
│ + puissance_w : float                  │
│ + tension_v : float                    │
└────────────────────────────────────────┘

correspondant en table SQL:

``` 
CREATE TABLE IF NOT EXISTS telemetrie_energie (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    sensor_id TEXT NOT NULL,
    timestamp DATETIME NOT NULL,
    puissance_w REAL NOT NULL,
    tension_v REAL NOT NULL
);

N'est-ce pas c'est simple ?

Il faut connaître les types seulement.


### 📚 Les Types de Données en SQL

En SQL, le type de donnée définit la nature des valeurs qu'une colonne a le droit d'accueillir et la manière dont la base de données les stocke sur disque.

---

#### 💡 1. Les 5 Types Natifs de SQLite

SQLite fonctionne avec un système de typage simplifié autour de 5 classes de stockage fondamentales :

| Type SQLite | Équivalent Python | Description | Exemple Métier (IoT Énergétique) |
| :--- | :--- | :--- | :--- |
| **`INTEGER`** | `int` | Nombre entier signé (stocké sur 1, 2, 3, 4 ou 8 octets selon la valeur). | `id` (1, 2, 3), `nbr_cycles` |
| **`REAL`** | `float` | Nombre à virgule flottante sur 8 octets (norme IEEE 64 bits). | `puissance_w` (1450.75), `tension_v` (230.2) |
| **`TEXT`** | `str` | Chaîne de caractères encodée en UTF-8 ou UTF-16. | `sensor_id` ("PANNEAU_01"), `emplacement` |
| **`BLOB`** | `bytes` | Binary Large Object : données binaires brutes conservées telles quelles. | Fichiers de firmware, images de caméras |
| **`NULL`** | `None` | Valeur absente, inconnue ou non renseignée. | Capteur hors ligne n'ayant renvoyé aucune mesure |

---

#### 🌐 2. Les Types SQL Standards (PostgreSQL, MySQL, Oracle)

Dans les bases de données relationnelles de production, les types sont plus spécialisés pour optimiser l'espace mémoire :

| Catégorie | Type SQL Standard | Description & Spécificité |
| :--- | :--- | :--- |
| **Nombres Entiers** | `SMALLINT` / `INT` / `BIGINT` | Entiers de petites à très grandes capacités (ex: `BIGINT` pour des milliards de lignes). |
| **Décimaux Précis** | `DECIMAL(p, s)` / `NUMERIC` | Stockage à virgule fixe à précision exacte (indispensable pour les transactions financières). |
| **Chaînes Fixes** | `CHAR(n)` | Chaîne de longueur fixe `n` (complétée automatiquement par des espaces). |
| **Chaînes Variables**| `VARCHAR(n)` | Chaîne de longueur variable limitée à `n` caractères max (ex: `VARCHAR(50)`). |
| **Temporalité** | `DATE` / `TIME` / `TIMESTAMP` | Distingue la date pure (`YYYY-MM-DD`), l'heure (`HH:MM:SS`) ou la combinaison des deux. |
| **Booleens** | `BOOLEAN` | Accepte uniquement `TRUE` ou `FALSE` (stocké sous forme `1` ou `0` sous SQLite). |

---

#### 🔍 3. Subtilités Techniques pour l'IoT

* **Gestion du temps dans SQLite :** SQLite ne possède pas de type `DATETIME` dédié. Par convention, on déclare la colonne `DATETIME`, mais SQLite la stocke sous forme de **`TEXT`** au format ISO 8601 (`"2026-08-30 14:30:00"`). Les fonctions SQL temporelles de SQLite savent interpréter ce format instantanément.
* **`REAL` (Flottant) vs `DECIMAL` (Fixe) :** Le type `REAL` utilise une approximation binaire. C'est parfait pour des mesures physiques (température, tension, puissance). En revanche, pour de la facturation d'énergie en euros, le type `DECIMAL` est obligatoire pour éliminer tout risque d'arrondi flottant.
* **Type vs Contrainte :** Le type indique *la nature* de la donnée (`REAL`), tandis que la contrainte impose *une règle de sécurité* (`NOT NULL`, `CHECK (puissance_w >= 0)`).

In [1]:
import os
import sqlite3

In [ ]:
# ==============================================================================
# 🟢 NIVEAU 01 : CONNEXION SQLITE ET CRÉATION DE LA TABLE DE TÉLÉMÉTRIE
# ==============================================================================
import os
import sqlite3

# Nom du fichier de base de données SQLite
DB_FILE = "iot_energie.db"

# Nettoyage automatique pour repartir sur une base propre lors des tests
if os.path.exists(DB_FILE):
    os.remove(DB_FILE)


# ------------------------------------------------------------------------------
# 🎯 QUÊTE 1 : À VOUS DE JOUER ! (Remplacez les ... par le code exact)
# ------------------------------------------------------------------------------

# ÉTAPE 1 : Établir la connexion au fichier SQLite
conn = sqlite3.connect(...)  # <--- Indiquer DB_FILE

# ÉTAPE 2 : Instancier le curseur SQL
cursor = conn.cursor()

# ÉTAPE 3 : Définir le schéma SQL de la table telemetrie_energie
sql_create_table = """
CREATE TABLE IF NOT EXISTS telemetrie_energie (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    sensor_id TEXT NOT NULL,
    timestamp DATETIME NOT NULL,
    puissance_w REAL NOT NULL,
    tension_v ...               -- <--- Indiquer le type et contrainte : REAL NOT NULL
);
"""

# ÉTAPE 4 : Exécuter la requête SQL de création de table
cursor.execute(...)  # <--- Transmettre sql_create_table

# ÉTAPE 5 : Valider et persister la transaction sur le disque
conn....()  # <--- Appeler la méthode commit()

# ------------------------------------------------------------------------------
# Vérification de la création de la table en base
# ------------------------------------------------------------------------------
cursor.execute(
    "SELECT name FROM sqlite_master WHERE type='table' AND name='telemetrie_energie';"
)
table_verif = cursor.fetchone()

# ÉTAPE 6 : Fermer la connexion
conn.close()


# ------------------------------------------------------------------------------
# 🧪 TESTS DE VALIDATION AUTOMATISÉS
# ------------------------------------------------------------------------------
assert os.path.exists(DB_FILE), (
    "Erreur : Le fichier 'iot_energie.db' n'a pas été créé."
)
assert table_verif is not None, (
    "Erreur : La table 'telemetrie_energie' n'existe pas dans la base."
)
assert table_verif[0] == "telemetrie_energie", (
    "Erreur : Le nom de la table est incorrect."
)

print(
    "🏆 Niveau 01 réussi ! La table 'telemetrie_energie' est opérationnelle."
)

## 🟢 Niveau 02 : Insertion de Données (INSERT INTO) & Sécurisation des Trames

Maintenant que le conteneur (la Table) est créé, nous allons y injecter nos premiers relevés de télémétrie IoT. 

---

### 💡 Correspondance POO vs SQL : De l'Instance à la Ligne

En POO, pour stocker plusieurs relevés, on ajoutait des objets dans une liste (`liste_mesures.append(obj)`). En SQL, on utilise la commande **`INSERT INTO`** pour persister chaque mesure sous forme de ligne dans la table.

```python
# En POO (RAM)
mesures = [
    ReleveEnergie("PANNEAU_SUD", "2026-08-30 14:00:00", 1250.5, 230.2),
    ReleveEnergie("EOLIENNE_01", "2026-08-30 14:01:00", 3100.0, 400.0),
]

##### En SQL (Stockage permanent) :

```sql
INSERT INTO telemetrie_energie (sensor_id, timestamp, puissance_w, tension_v)
VALUES ('PANNEAU_SUD', '2026-08-30 14:00:00', 1250.5, 230.2);

### 🔍 Les 3 Règles d'Or pour Insérer des Données IoT

* **1. Pas de valeur pour `id` :** La colonne `id` étant configurée en `AUTOINCREMENT`, SQLite génère automatiquement 1, 2, 3... Il ne faut **jamais** la passer dans la commande `INSERT`.
* **2. Utiliser les requêtes paramétrées (`?`) :** Ne JAMAIS utiliser la concaténation de chaînes (f-strings `f"INSERT... {valeur}"`). C'est la faille de sécurité majeure appelée **Injection SQL**, et cela provoque des erreurs de syntaxe sur les dates et guillemets.
* **3. Insertion unitaire (`execute`) vs par lot (`executemany`) :**
  * `cursor.execute(sql, tuple)` : Insère un seul relevé.
  * `cursor.executemany(sql, liste_de_tuples)` : Insère un lot de trames IoT en une seule transaction ultra-rapide.

---

### ⚡ Comparatif de Sécurité

```python
# ❌ MAUVAISE PRATIQUE (Risque d'injection SQL & erreurs de syntaxe) :
cursor.execute(
    f"INSERT INTO telemetrie_energie VALUES (NULL, '{sensor}', '{time}', {p},"
    f" {v})"
)

# ✅ BONNE PRATIQUE (Requête paramétrée sécurisée avec placeholders ?) :
cursor.execute(
    "INSERT INTO telemetrie_energie (sensor_id, timestamp, puissance_w,"
    " tension_v) VALUES (?, ?, ?, ?)",
    (sensor, time, p, v),
)

In [ ]:
# ==============================================================================
# 🟢 NIVEAU 02 : INSERTION DE TÉLÉMÉTRIE & LECTURE INITIALE (SELECT)
# ==============================================================================
import os
import sqlite3

DB_FILE = "iot_energie.db"

# 🔑 ÉTAPE OBLIGATOIRE : Réouvrir la connexion et le curseur fermés au précedent
conn = sqlite3.connect(DB_FILE)
cursor = conn.cursor()

# ------------------------------------------------------------------------------
# 🎯 QUÊTE 2 : À VOUS DE JOUER !
# ------------------------------------------------------------------------------

# ÉTAPE 1 : Trame unitaire issue d'un capteur Solaire
trame_unique = ("SOLAR_PANEL_01", "2026-08-30 14:00:00", 1450.75, 230.1)

sql_insert_unitaire = """
INSERT INTO telemetrie_energie (sensor_id, timestamp, puissance_w, tension_v)
VALUES (?, ?, ?, ?);
"""

# Insertion de la trame unique 
cursor...(sql_insert_unitaire, trame_unique)


# ÉTAPE 2 : Lot de trames télémetriques issues du réseau Smart Grid
lot_trames = [
    ("SOLAR_PANEL_01", "2026-08-30 14:15:00", 1520.00, 230.4),
    ("WIND_TURBINE_02", "2026-08-30 14:15:00", 3400.50, 400.1),
    ("BATTERY_BANK_01", "2026-08-30 14:15:00", -450.00, 48.2),
    ("WIND_TURBINE_02", "2026-08-30 14:30:00", 3100.20, 399.8),
]

# Insérer le lot complet
cursor...( ..., lot_trames) 

# ÉTAPE 3 : Persister l'écriture sur le disque
conn...()

# ÉTAPE 4 : Vérifier l'insertion avec SELECT
sql_select_all = "SELECT * FROM telemetrie_energie;"
cursor.execute(sql_select_all)

toutes_les_lignes = cursor.fetchall()

# Fermeture propre à la toute fin de la cellule
conn....()


# ------------------------------------------------------------------------------
# 🧪 TESTS DE VALIDATION AUTOMATISÉS
# ------------------------------------------------------------------------------
assert len(toutes_les_lignes) == 5, (
    f"Erreur : Attendu 5 lignes, trouvé {len(toutes_les_lignes)}."
)

premier_id = toutes_les_lignes[0][0]
dernier_id = toutes_les_lignes[-1][0]

assert premier_id == 1, (
    f"Erreur : Le premier ID doit être 1, trouvé {premier_id}."
)
assert dernier_id == 5, (
    f"Erreur : Le dernier ID doit être 5, trouvé {dernier_id}."
)

print(
    "🏆 Niveau 02 réussi ! 5 trames de télémétrie insérées et lues avec"
    " succès."
)

## 🟢 Niveau 03 : Requêtes de Filtrage & Tri (SELECT, WHERE, ORDER BY, LIMIT)

Extaire la totalité d'une base de données en mémoire RAM devient impossible à l'échelle industrielle. En SQL, le filtrage s'effectue directement au niveau du moteur de stockage via des requêtes ciblées.

---

### 💡 Synthèse des Mots-Clés de Sélection & Filtrage

| Clause SQL | Rôle Métier IoT | Exemple d'utilisation |
| :--- | :--- | :--- |
| **`SELECT col1, col2`** | Extrait uniquement les colonnes utiles (évite de charger `id` ou des métadonnées inutiles). | `SELECT timestamp, puissance_w` |
| **`WHERE condition`** | Filtre les lignes selon un ou plusieurs critères (opérateurs `>`, `<`, `=`, `!=`, `AND`, `OR`). | `WHERE puissance_w > 1500` |
| **`ORDER BY col ASC/DESC`** | Trie les résultats par ordre croissant (`ASC`) ou décroissant (`DESC`). | `ORDER BY puissance_w DESC` |
| **`LIMIT n`** | Restreint le résultat aux $n$ premiers enregistrements (ex: Top 3 des pics). | `LIMIT 3` |

---

### 🔍 Bonnes Pratiques pour les Filtres Dynamo-Saisis

Tout comme pour l'insertion (`INSERT INTO`), l'utilisation de variables Python dans une clause `WHERE` doit **toujours utiliser des requêtes paramétrées avec `?`** pour éviter toute injection SQL.

```python
# ❌ À ÉVITER :
cursor.execute(f"SELECT * FROM telemetrie_energie WHERE sensor_id = '{nom_capteur}'")

# ✅ SÉCURISÉ :
cursor.execute("SELECT * FROM telemetrie_energie WHERE sensor_id = ?", (nom_capteur,))

In [ ]:
DB_FILE = "iot_energie.db"

# 🔑 ÉTAPE OBLIGATOIRE : Réouvrir la connexion et le curseur
conn = sqlite3.connect(DB_FILE)
cursor = conn.cursor()

# Initialisation et re-remplissage d'un jeu de données de test complet
cursor.execute("""
CREATE TABLE IF NOT EXISTS telemetrie_energie (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    sensor_id TEXT NOT NULL,
    timestamp DATETIME NOT NULL,
    puissance_w REAL NOT NULL,
    tension_v REAL NOT NULL
);
""")
cursor.execute("DELETE FROM telemetrie_energie;")

donnees_test = [
    ("SOLAR_PANEL_01", "2026-08-30 14:00:00", 1450.75, 230.1),
    ("SOLAR_PANEL_01", "2026-08-30 14:15:00", 1520.00, 230.4),
    ("WIND_TURBINE_02", "2026-08-30 14:15:00", 3400.50, 400.1),
    ("BATTERY_BANK_01", "2026-08-30 14:15:00", -450.00, 48.2),
    ("WIND_TURBINE_02", "2026-08-30 14:30:00", 3100.20, 399.8),
    ("SOLAR_PANEL_01", "2026-08-30 14:30:00", 980.10, 229.8),
    ("BATTERY_BANK_01", "2026-08-30 14:30:00", 1200.00, 49.0),
]

cursor.executemany(
    "INSERT INTO telemetrie_energie (sensor_id, timestamp, puissance_w, tension_v) VALUES (?, ?, ?, ?);",
    donnees_test,
)
conn.commit()


# ------------------------------------------------------------------------------
# 🎯 QUÊTE 3 : À VOUS DE JOUER ! (Remplacez les ... par le code exact)
# ------------------------------------------------------------------------------

# ÉTAPE 1 : Extraire uniquement (timestamp, puissance_w) pour le capteur "SOLAR_PANEL_01"
sql_exo1 = """
SELECT timestamp, puissance_w 
FROM telemetrie_energie 
WHERE sensor_id = ?;
"""
cursor.execute(sql_exo1, ("SOLAR_PANEL_01",))
releves_solaire = cursor.fetchall()


# ÉTAPE 2 : Sélectionner tous les enregistrements où la puissance dépasse 1500 Watts
sql_exo2 = """
SELECT * FROM telemetrie_energie 
WHERE puissance_w > ...;             -- <--- Indiquer le seuil numérique (1500)
"""
cursor.execute(sql_exo2)
fortes_puissances = cursor.fetchall()


# ÉTAPE 3 : Identifier les 2 plus importants pics de puissance de tout le réseau
sql_exo3 = """
SELECT sensor_id, puissance_w 
FROM telemetrie_energie 
ORDER BY puissance_w ...             -- <--- Préciser le sens du tri : DESC (décroissant)
LIMIT 2;
"""
cursor.execute(sql_exo3)
top2_pics = cursor.fetchall()

# Fermeture de la connexion
conn.close()


# ------------------------------------------------------------------------------
# 🧪 TESTS DE VALIDATION AUTOMATISÉS
# ------------------------------------------------------------------------------
assert len(releves_solaire) == 3, (
    f"Erreur Exo 1 : Attendu 3 relevés solaires, trouvé {len(releves_solaire)}."
)
assert len(releves_solaire[0]) == 2, (
    "Erreur Exo 1 : La requête doit retourner exactement 2 colonnes (timestamp, puissance_w)."
)

assert len(fortes_puissances) == 3, (
    f"Erreur Exo 2 : Attendu 3 mesures > 1500W, trouvé {len(fortes_puissances)}."
)

assert len(top2_pics) == 2, (
    f"Erreur Exo 3 : Attendu 2 résultats (LIMIT 2), trouvé {len(top2_pics)}."
)
assert top2_pics[0][1] == 3400.50, (
    f"Erreur Exo 3 : Le pic maximal doit être 3400.50, trouvé {top2_pics[0][1]}."
)

print(
    "🏆 Niveau 03 réussi ! Maîtrise des filtres WHERE, tris ORDER BY et limites d'extraction."
)



## 🟢 Niveau 04 : Agrégations & Calculs Statistiques (COUNT, AVG, MAX, MIN, SUM)

Au lieu de charger des millions de lignes dans la mémoire Python pour calculer des moyennes ou des totaux, le moteur SQL effectue ces calculs directement sur le disque dur. C'est le rôle des **fonctions d'agrégation**.

---

### 💡 Les Fonctions d'Agrégation Essentielles

Une fonction d'agrégation prend un ensemble de valeurs dans une colonne et retourne une **seule valeur résumée**.

| Fonction SQL | Rôle Métier IoT | Exemple d'utilisation |
| :--- | :--- | :--- |
| **`COUNT(*)`** | Compte le nombre total de trames enregistrées. | `SELECT COUNT(*) FROM telemetrie_energie` |
| **`AVG(colonne)`** | Calcule la moyenne (ex: tension moyenne du réseau). | `SELECT AVG(tension_v) FROM telemetrie_energie` |
| **`MAX(colonne)`** | Identifie la valeur maximale (ex: pic de puissance). | `SELECT MAX(puissance_w) FROM telemetrie_energie` |
| **`MIN(colonne)`** | Identifie la valeur minimale (ex: tension la plus basse). | `SELECT MIN(tension_v) FROM telemetrie_energie` |
| **`SUM(colonne)`** | Calcule la somme globale d'une métrique. | `SELECT SUM(puissance_w) FROM telemetrie_energie` |
| **`ROUND(valeur, n)`** | Arrondit le résultat à $n$ décimales. | `SELECT ROUND(AVG(puissance_w), 2)` |

---

### 🔍 Combiner Agrégation et Filtrage (`WHERE`)

Il est tout à fait possible de filtrer les données *avant* de calculer une statistique en ajoutant une clause `WHERE`. Par exemple, pour calculer la puissance moyenne uniquement durant la journée ou uniquement pour les panneaux solaires :

```sql
SELECT ROUND(AVG(puissance_w), 2) AS puissance_moyenne_solaire
FROM telemetrie_energie
WHERE sensor_id = 'SOLAR_PANEL_01';

```

```




In [ ]:

DB_FILE = "iot_energie.db"

# 🔑 ÉTAPE OBLIGATOIRE : Réouvrir la connexion et le curseur
conn = sqlite3.connect(DB_FILE)
cursor = conn.cursor()

# Initialisation et insertion d'un jeu de données de test complet
cursor.execute("""
CREATE TABLE IF NOT EXISTS telemetrie_energie (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    sensor_id TEXT NOT NULL,
    timestamp DATETIME NOT NULL,
    puissance_w REAL NOT NULL,
    tension_v REAL NOT NULL
);
""")
cursor.execute("DELETE FROM telemetrie_energie;")

donnees_test = [
    ("SOLAR_PANEL_01", "2026-08-30 14:00:00", 1450.75, 230.1),
    ("SOLAR_PANEL_01", "2026-08-30 14:15:00", 1520.00, 230.4),
    ("WIND_TURBINE_02", "2026-08-30 14:15:00", 3400.50, 400.1),
    ("BATTERY_BANK_01", "2026-08-30 14:15:00", -450.00, 48.2),
    ("WIND_TURBINE_02", "2026-08-30 14:30:00", 3100.20, 399.8),
    ("SOLAR_PANEL_01", "2026-08-30 14:30:00", 980.10, 229.8),
    ("BATTERY_BANK_01", "2026-08-30 14:30:00", 1200.00, 49.0),
]

cursor.executemany(
    "INSERT INTO telemetrie_energie (sensor_id, timestamp, puissance_w, tension_v) VALUES (?, ?, ?, ?);",
    donnees_test,
)
conn.commit()


# ------------------------------------------------------------------------------
# 🎯 QUÊTE 4 : À VOUS DE JOUER ! (Remplacez les ... par le code exact)
# ------------------------------------------------------------------------------

# ÉTAPE 1 : Compter le nombre total de trames et trouver le pic de puissance global
sql_exo1 = """
SELECT COUNT(*), ... (puissance_w)   -- <--- Indiquer la fonction d'agrégation du maximum (MAX)
FROM telemetrie_energie;
"""
cursor.execute(sql_exo1)
total_trames, pic_max = cursor.fetchone()


# ÉTAPE 2 : Calculez la puissance moyenne du capteur 'SOLAR_PANEL_01', arrondie à 2 décimales
sql_exo2 = """
SELECT ROUND( ... (puissance_w), 2)  -- <--- Indiquer la fonction de moyenne (AVG)
FROM telemetrie_energie 
WHERE sensor_id = 'SOLAR_PANEL_01';
"""
cursor.execute(sql_exo2)
moyenne_solaire = cursor.fetchone()[0]


# ÉTAPE 3 : Calculez la somme totale de puissance produite uniquement par les sources génératrices (puissance_w > 0)
sql_exo3 = """
SELECT ... (puissance_w)             -- <--- Indiquer la fonction de somme (SUM)
FROM telemetrie_energie 
WHERE puissance_w > 0;
"""
cursor.execute(sql_exo3)
puissance_totale_produite = cursor.fetchone()[0]

# Fermeture propre de la connexion
conn.close()


# ------------------------------------------------------------------------------
# 🧪 TESTS DE VALIDATION AUTOMATISÉS
# ------------------------------------------------------------------------------
assert total_trames == 7, (
    f"Erreur Exo 1 : Attendu 7 trames au total, trouvé {total_trames}."
)
assert pic_max == 3400.50, (
    f"Erreur Exo 1 : Pic max attendu 3400.50, trouvé {pic_max}."
)

assert moyenne_solaire == 1316.95, (
    f"Erreur Exo 2 : Moyenne solaire attendue 1316.95, trouvée {moyenne_solaire}."
)

assert puissance_totale_produite == 11651.55, (
    f"Erreur Exo 3 : Somme attendue 11651.55, trouvée {puissance_totale_produite}."
)

print(
    "🏆 Niveau 04 réussi ! Maîtrise des calculs statistiques agrégés (COUNT, AVG, MAX, SUM)."
)



## 🟢 Niveau 05 : Groupement & Filtrage d'Agrégat (GROUP BY & HAVING)

Dans le niveau précédent, nous avons calculé des statistiques globales sur toute la table. En IoT, nous devons souvent calculer ces statistiques **par équipement** (ex: la puissance moyenne de chaque capteur individuellement). C'est le rôle de la clause **`GROUP BY`**.

---

### 💡 Le Duo `GROUP BY` & `HAVING`

| Clause SQL | Rôle Métier IoT | Exemple d'utilisation |
| :--- | :--- | :--- |
| **`GROUP BY colonne`** | Découpe le tableau en sous-groupes (un groupe par valeur unique de la colonne). | `GROUP BY sensor_id` |
| **`HAVING condition`** | Filtre les groupes **après** agrégation (équivalent du `WHERE` mais appliqué aux résultats calculés). | `HAVING AVG(puissance_w) > 1000` |

---

### 🔍 La Différence Cruciale : `WHERE` vs `HAVING`

* **`WHERE` (Avant agrégation) :** Filtre les **lignes individuelles** du tableau avant de faire les calculs. Il ne peut pas contenir de fonction d'agrégation (interdit d'écrire `WHERE AVG(puissance_w) > 1000`).
* **`HAVING` (Après agrégation) :** Filtre les **groupes résumés** calculés par `GROUP BY`.

```text
Ordre d'exécution SQL :
1. FROM               ──> Sélection de la table
2. WHERE              ──> Filtrage des lignes brutes
3. GROUP BY           ──> Regroupement des lignes
4. HAVING             ──> Filtrage des groupes calculés
5. SELECT             ──> Projection des colonnes et calculs
6. ORDER BY / LIMIT   ──> Tri et limitation des résultats

```

```


In [ ]:


# ==============================================================================
# 🟢 NIVEAU 05 : GROUPEMENT ET FILTRAGE D'AGRÉGATS (GROUP BY & HAVING)
# ==============================================================================
import os
import sqlite3

DB_FILE = "iot_energie.db"

# 🔑 ÉTAPE OBLIGATOIRE : Réouvrir la connexion et le curseur
conn = sqlite3.connect(DB_FILE)
cursor = conn.cursor()

# Initialisation et insertion des trames de test
cursor.execute("""
CREATE TABLE IF NOT EXISTS telemetrie_energie (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    sensor_id TEXT NOT NULL,
    timestamp DATETIME NOT NULL,
    puissance_w REAL NOT NULL,
    tension_v REAL NOT NULL
);
""")
cursor.execute("DELETE FROM telemetrie_energie;")

donnees_test = [
    ("SOLAR_PANEL_01", "2026-08-30 14:00:00", 1450.75, 230.1),
    ("SOLAR_PANEL_01", "2026-08-30 14:15:00", 1520.00, 230.4),
    ("WIND_TURBINE_02", "2026-08-30 14:15:00", 3400.50, 400.1),
    ("BATTERY_BANK_01", "2026-08-30 14:15:00", -450.00, 48.2),
    ("WIND_TURBINE_02", "2026-08-30 14:30:00", 3100.20, 399.8),
    ("SOLAR_PANEL_01", "2026-08-30 14:30:00", 980.10, 229.8),
    ("BATTERY_BANK_01", "2026-08-30 14:30:00", 1200.00, 49.0),
]

cursor.executemany(
    "INSERT INTO telemetrie_energie (sensor_id, timestamp, puissance_w, tension_v) VALUES (?, ?, ?, ?);",
    donnees_test,
)
conn.commit()


# ------------------------------------------------------------------------------
# 🎯 QUÊTE 5 : À VOUS DE JOUER ! (Remplacez les ... par le code exact)
# ------------------------------------------------------------------------------

# ÉTAPE 1 : Calculez le nombre de trames et la puissance moyenne pour CHAQUE capteur
sql_exo1 = """
SELECT sensor_id, COUNT(*), ROUND(AVG(puissance_w), 2)
FROM telemetrie_energie
GROUB BY sensor_id;                    -- <--- Compléter avec le mot-clé de regroupement (GROUP)
"""
cursor.execute(sql_exo1)
stats_par_capteur = cursor.fetchall()


# ÉTAPE 2 : Conservez uniquement les capteurs dont la puissance moyenne dépasse 1000 Watts
sql_exo2 = """
SELECT sensor_id, ROUND(AVG(puissance_w), 2) AS puissance_moyenne
FROM telemetrie_energie
GROUP BY sensor_id
HAVING AVG(puissance_w) > 1000;         -- <--- Compléter avec le mot-clé de filtre sur agrégat (HAVING)
"""
cursor.execute(sql_exo2)
capteurs_fortes_moyennes = cursor.fetchall()


# ÉTAPE 3 : Filtrer d'abord les valeurs positives (WHERE > 0), grouper par capteur, et trier par puissance moyenne décroissante
sql_exo3 = """
SELECT sensor_id, ROUND(AVG(puissance_w), 2) AS puissance_moyenne
FROM telemetrie_energie
WHERE puissance_w > 0
GROUP BY sensor_id
ORDER BY puissance_moyenne ...;      -- <--- Indiquer le sens du tri : DESC (décroissant)
"""
cursor.execute(sql_exo3)
classement_générateurs = cursor.fetchall()

# Fermeture propre de la connexion
conn.close()


# ------------------------------------------------------------------------------
# 🧪 TESTS DE VALIDATION AUTOMATISÉS
# ------------------------------------------------------------------------------
assert len(stats_par_capteur) == 3, (
    f"Erreur Exo 1 : Attendu 3 capteurs distincts, trouvé {len(stats_par_capteur)}."
)

assert len(capteurs_fortes_moyennes) == 2, (
    f"Erreur Exo 2 : Attendu 2 capteurs avec moyenne > 1000W, trouvé {len(capteurs_fortes_moyennes)}."
)

assert classement_générateurs[0][0] == "WIND_TURBINE_02", (
    f"Erreur Exo 3 : Le premier équipement doit être WIND_TURBINE_02, trouvé {classement_générateurs[0][0]}."
)
assert classement_générateurs[0][1] == 3250.35, (
    f"Erreur Exo 3 : La moyenne maximale doit être 3250.35, trouvée {classement_générateurs[0][1]}."
)

print(
    "🏆 Niveau 05 réussi ! Maîtrise des regroupements GROUP BY et filtres d'agrégats HAVING."
)


## 🟢 Niveau 06 : Traitement des Séries Temporelles SQL (date, strftime)

Dans un système IoT de gestion de l'énergie, la variable temporelle est centrale. Nous devons analyser la production heure par heure, comparer des journées, ou détecter les pics de consommation selon la période du jour.

---

### 💡 La Gestion du Temps dans SQLite

Comme vu précédemment, SQLite ne possède pas de type `DATETIME` natif séparé : il stocke les dates sous forme de chaînes de caractères au format ISO 8601 (`"YYYY-MM-DD HH:MM:SS"`). SQLite fournit des fonctions intégrées puissantes pour manipuler ces chaînes temporelles :

| Fonction SQL | Rôle Métier IoT | Exemple d'utilisation |
| :--- | :--- | :--- |
| **`DATE(timestamp)`** | Extrait uniquement la partie Date (`"YYYY-MM-DD"`). | `SELECT DATE(timestamp)` |
| **`TIME(timestamp)`** | Extrait uniquement la partie Heure (`"HH:MM:SS"`). | `SELECT TIME(timestamp)` |
| **`strftime(format, timestamp)`** | Formate et extrait des composantes précises (`%H` heure, `%d` jour, `%m` mois, `%Y` année). | `SELECT strftime('%H', timestamp)` |

---

### 🔍 Formats Courants de `strftime`

* `%Y` : Année sur 4 chiffres (ex: `2026`)
* `%m` : Mois sur 2 chiffres (`01` à `12`)
* `%d` : Jour du mois (`01` à `31`)
* `%H` : Heure au format 24h (`00` à `23`)
* `%M` : Minutes (`00` à `59`)

```sql
-- Exemple : Calculer la puissance moyenne du réseau heure par heure (tous jours confondus)
SELECT strftime('%H', timestamp) AS heure, ROUND(AVG(puissance_w), 2) AS moyenne_w
FROM telemetrie_energie
GROUP BY heure
ORDER BY heure ASC;

```

```



In [ ]:
DB_FILE = "iot_energie.db"

# 🔑 ÉTAPE OBLIGATOIRE : Réouvrir la connexion et le curseur
conn = sqlite3.connect(DB_FILE)
cursor = conn.cursor()

# Initialisation et re-remplissage avec une série temporelle sur 2 jours
cursor.execute("""
CREATE TABLE IF NOT EXISTS telemetrie_energie (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    sensor_id TEXT NOT NULL,
    timestamp DATETIME NOT NULL,
    puissance_w REAL NOT NULL,
    tension_v REAL NOT NULL
);
""")
cursor.execute("DELETE FROM telemetrie_energie;")

donnees_temporelles = [
    ("SOLAR_PANEL_01", "2026-08-30 08:00:00", 450.00, 228.5),
    ("SOLAR_PANEL_01", "2026-08-30 12:00:00", 1850.50, 231.0),
    ("SOLAR_PANEL_01", "2026-08-30 14:00:00", 1600.00, 230.5),
    ("SOLAR_PANEL_01", "2026-08-31 08:00:00", 420.00, 229.0),
    ("SOLAR_PANEL_01", "2026-08-31 12:00:00", 1910.00, 231.5),
    ("WIND_TURBINE_02", "2026-08-30 08:00:00", 2100.00, 400.0),
    ("WIND_TURBINE_02", "2026-08-30 12:00:00", 2500.00, 401.0),
]

cursor.executemany(
    "INSERT INTO telemetrie_energie (sensor_id, timestamp, puissance_w, tension_v) VALUES (?, ?, ?, ?);",
    donnees_temporelles,
)
conn.commit()


# ------------------------------------------------------------------------------
# 🎯 QUÊTE 6 : À VOUS DE JOUER ! (Remplacez les ... par le code exact)
# ------------------------------------------------------------------------------

# ÉTAPE 1 : Regroupez la production totale par JOUR (en utilisant DATE(timestamp))
sql_exo1 = """
SELECT ...(timestamp) AS jour, SUM(puissance_w)   -- <--- Indiquer la fonction SQL qui extrait la date (DATE)
FROM telemetrie_energie
GROUP BY jour
ORDER BY jour ASC;
"""
cursor.execute(sql_exo1)
production_par_jour = cursor.fetchall()


# ÉTAPE 2 : Filtrez uniquement les trames enregistrées pendant le creux Solaire de mi-journée (entre 11h et 13h inclus)
sql_exo2 = """
SELECT sensor_id, timestamp, puissance_w
FROM telemetrie_energie
WHERE strftime('%H', timestamp) BETWEEN ... AND ...;  -- <--- Indiquer les heures limites sous forme de texte ('11' et '13')
"""
cursor.execute(sql_exo2)
mesures_mi_journee = cursor.fetchall()


# ÉTAPE 3 : Calculez la puissance moyenne par HEURE de la journée (%H), triée de la plus tôt à la plus tardive
sql_exo3 = """
SELECT strftime('...', timestamp) AS heure, ROUND(AVG(puissance_w), 2)  -- <--- Indiquer le format d'heure (%H)
FROM telemetrie_energie
GROUP BY heure
ORDER BY heure ASC;
"""
cursor.execute(sql_exo3)
moyennes_horaires = cursor.fetchall()

# Fermeture propre de la connexion
conn.close()


# ------------------------------------------------------------------------------
# 🧪 TESTS DE VALIDATION AUTOMATISÉS
# ------------------------------------------------------------------------------
assert len(production_par_jour) == 2, (
    f"Erreur Exo 1 : Attendu 2 jours distincts, trouvé {len(production_par_jour)}."
)
assert production_par_jour[0][0] == "2026-08-30", (
    f"Erreur Exo 1 : Le premier jour doit être '2026-08-30', trouvé {production_par_jour[0][0]}."
)

assert len(mesures_mi_journee) == 3, (
    f"Erreur Exo 2 : Attendu 3 mesures entre 11h et 13h, trouvé {len(mesures_mi_journee)}."
)

assert len(moyennes_horaires) == 3, (
    f"Erreur Exo 3 : Attendu 3 créneaux horaires distincts (08h, 12h, 14h), trouvé {len(moyennes_horaires)}."
)
assert moyennes_horaires[1][0] == "12", (
    f"Erreur Exo 3 : Le second créneau doit être l'heure '12', trouvé {moyennes_horaires[1][0]}."
)

print(
    "🏆 Niveau 06 réussi ! Maîtrise de l'analyse temporelle avec DATE et strftime."
)



## 🟢 Niveau 07 : Maintenance & Nettoyage des Données (UPDATE & DELETE)

Dans la vie d'un réseau IoT, les capteurs subissent parfois des dysfonctionnements : erreurs de calibrage, trames corrompues ou pannes matérielles. En SQL, la maintenance et le nettoyage du stockage s'effectuent via les commandes **`UPDATE`** (modification) et **`DELETE`** (suppression).

---

### 💡 Synthèse des Mots-Clés de Modification et Suppression

| Clause SQL | Rôle Métier IoT | Exemple d'utilisation |
| :--- | :--- | :--- |
| **`UPDATE table SET col = val`** | Modifie la valeur d'une ou plusieurs colonnes sur les lignes ciblées. | `UPDATE telemetrie_energie SET tension_v = 230.0` |
| **`DELETE FROM table`** | Supprime définitivement des lignes de la base de données. | `DELETE FROM telemetrie_energie WHERE tension_v = 0` |

---

### ⚠️ La Règle d'Or en Production : N'oubliez JAMAIS le `WHERE` !

* Sans clause `WHERE`, une commande `UPDATE` **modifie la totalité des lignes** de votre table.
* Sans clause `WHERE`, une commande `DELETE FROM` **efface l'intégralité du contenu** de votre table.

```python
# ❌ ERREUR FATALE (Écrase toute la base avec la valeur 0) :
cursor.execute("UPDATE telemetrie_energie SET puissance_w = 0;")

# ✅ PRATIQUE SÉCURISÉE (Cible uniquement le capteur défectueux) :
cursor.execute(
    "UPDATE telemetrie_energie SET puissance_w = 0 WHERE sensor_id = ?",
    (capteur_panne,),
)

```

```


In [ ]:
DB_FILE = "iot_energie.db"

# 🔑 ÉTAPE OBLIGATOIRE : Réouvrir la connexion et le curseur
conn = sqlite3.connect(DB_FILE)
cursor = conn.cursor()

# Initialisation et re-remplissage avec des données contenant des anomalies IoT
cursor.execute("""
CREATE TABLE IF NOT EXISTS telemetrie_energie (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    sensor_id TEXT NOT NULL,
    timestamp DATETIME NOT NULL,
    puissance_w REAL NOT NULL,
    tension_v REAL NOT NULL
);
""")
cursor.execute("DELETE FROM telemetrie_energie;")

donnees_anomales = [
    (
        "SOLAR_PANEL_01",
        "2026-08-30 14:00:00",
        1450.75,
        220.0,
    ),  # Tension sous-estimée (erreur étalonnage)
    (
        "SOLAR_PANEL_01",
        "2026-08-30 14:15:00",
        1520.00,
        220.0,
    ),  # Tension sous-estimée (erreur étalonnage)
    (
        "SENSOR_BUG_99",
        "2026-08-30 14:20:00",
        -9999.00,
        0.0,
    ),  # Trame corrompue (à supprimer)
    ("WIND_TURBINE_02", "2026-08-30 14:30:00", 3100.20, 399.8),
    (
        "SENSOR_BUG_99",
        "2026-08-30 14:35:00",
        -9999.00,
        0.0,
    ),  # Trame corrompue (à supprimer)
]

cursor.executemany(
    "INSERT INTO telemetrie_energie (sensor_id, timestamp, puissance_w,"
    " tension_v) VALUES (?, ?, ?, ?);",
    donnees_anomales,
)
conn.commit()


# ------------------------------------------------------------------------------
# 🎯 QUÊTE 7 : À VOUS DE JOUER ! (Remplacez les ... par le code exact)
# ------------------------------------------------------------------------------

# ÉTAPE 1 : Étalonnage — Corriger la tension du capteur 'SOLAR_PANEL_01' en la fixant à 230.0 V
sql_exo1 = """
UPDATE telemetrie_energie
... tension_v = 230.0                -- <--- Indiquer le mot-clé d'affectation (SET)
WHERE sensor_id = 'SOLAR_PANEL_01';
"""
cursor.execute(sql_exo1)


# ÉTAPE 2 : Purge — Supprimer définitivement toutes les trames corrompues envoyées par 'SENSOR_BUG_99'
sql_exo2 = """
... FROM telemetrie_energie          -- <--- Indiquer la commande de suppression (DELETE)
WHERE sensor_id = ?;
"""
cursor.execute(sql_exo2, ("SENSOR_BUG_99",))


# ÉTAPE 3 : Valider les modifications et vérifier les données restantes
conn.commit()

cursor.execute(
    "SELECT sensor_id, tension_v FROM telemetrie_energie WHERE sensor_id ="
    " 'SOLAR_PANEL_01';"
)
solaire_corriges = cursor.fetchall()

cursor.execute("SELECT COUNT(*) FROM telemetrie_energie;")
nombre_lignes_restantes = cursor.fetchone()[0]

# Fermeture propre de la connexion
conn.close()


# ------------------------------------------------------------------------------
# 🧪 TESTS DE VALIDATION AUTOMATISÉS
# ------------------------------------------------------------------------------
assert nombre_lignes_restantes == 3, (
    "Erreur Exo 2 : Attendu 3 lignes restantes après purge, trouvé"
    f" {nombre_lignes_restantes}."
)

assert (
    len(solaire_corriges) == 2
    and solaire_corriges[0][1] == 230.0
    and solaire_corriges[1][1] == 230.0
), (
    "Erreur Exo 1 : La tension des relevés de SOLAR_PANEL_01 doit être réétalonnée"
    " à 230.0 V."
)

print(
    "🏆 Niveau 07 réussi ! Maîtrise des mises à jour UPDATE et des purges"
    " DELETE."
)



## 🟢 Niveau 08 : Modélisation Relationnelle & Joins (JOIN & Clés Étrangères)

Dans un système IoT réel, stocker l'emplacement, le modèle ou le fabricant à côté de chaque mesure dans la table `telemetrie_energie` provoquerait une massive duplication de données. Pour respecter les règles de **normalisation SQL**, on sépare la structure en 2 tables distinctes :

1. **`capteurs` (Table Statique / Métadonnées)** : Stocke la fiche d'identité matérielle de chaque équipement (emplacement, type, puissance nominale).
2. **`telemetrie_energie` (Table Dynamique / Séries Temporelles)** : Stocke uniquement les flux de mesures associées au code du capteur.

---

### 💡 La Clé Étrangère (`FOREIGN KEY`) et le `JOIN`

La liaison entre ces deux univers s'effectue via une **Clé Étrangère** (`FOREIGN KEY`), connectant la colonne `sensor_id` de la télémétrie au code unique `sensor_code` du capteur.

| Commande / Concept | Rôle Métier IoT | Syntaxe SQL |
| :--- | :--- | :--- |
| **`FOREIGN KEY`** | Garantit l'intégrité référentielle : interdit d'insérer des mesures pour un capteur qui n'existe pas dans la table `capteurs`. | `FOREIGN KEY (sensor_id) REFERENCES capteurs(sensor_code)` |
| **`INNER JOIN`** | Combine les lignes de deux tables en associant les enregistrements qui partagent la même clé. | `FROM telemetrie_energie JOIN capteurs ON sensor_id = sensor_code` |

---

### 🔍 Représentation Visuelle de la Jonction

```text
 Table [ capteurs ]                             Table [ telemetrie_energie ]
┌─────────────┬─────────────┐                  ┌────┬─────────────┬─────────────┐
│ sensor_code │ emplacement │ ◄──┐ (Jonction)  │ id │ sensor_id   │ puissance_w │
├─────────────┼─────────────┤    └───────────  ├────┼─────────────┼─────────────┤
│ SOLAR_01    │ Zone Nord   │                  │ 1  │ SOLAR_01    │ 1450.75     │
│ WIND_02     │ Ferme Est   │                  │ 2  │ WIND_02     │ 3400.50     │
└─────────────┴─────────────┘                  └────┴─────────────┴─────────────┘

```

```


In [ ]:
DB_FILE = "iot_energie.db"

# 🔑 ÉTAPE OBLIGATOIRE : Réouvrir la connexion et le curseur
conn = sqlite3.connect(DB_FILE)
cursor = conn.cursor()

# Activer explicitement le support des Clés Étrangères dans SQLite
cursor.execute("PRAGMA foreign_keys = ON;")

# Nettoyage et création du schéma à 2 tables
cursor.execute("DROP TABLE IF EXISTS telemetrie_energie;")
cursor.execute("DROP TABLE IF EXISTS capteurs;")

# 1. Table des métadonnées capteurs
cursor.execute("""
CREATE TABLE capteurs (
    sensor_code TEXT PRIMARY KEY,
    emplacement TEXT NOT NULL,
    type_equipement TEXT NOT NULL
);
""")

# 2. Table de télémétrie avec Clé Étrangère
cursor.execute("""
CREATE TABLE telemetrie_energie (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    sensor_id TEXT NOT NULL,
    timestamp DATETIME NOT NULL,
    puissance_w REAL NOT NULL,
    tension_v REAL NOT NULL,
    FOREIGN KEY (sensor_id) REFERENCES capteurs(sensor_code)
);
""")

# Insertion du parc de capteurs (Métadonnées)
parc_capteurs = [
    ("SOLAR_PANEL_01", "Toiture Bâtiment A", "Solaire"),
    ("WIND_TURBINE_02", "Zone Eolienne Est", "Eolien"),
    ("BATTERY_BANK_01", "Local Technique 01", "Stockage"),
]
cursor.executemany(
    "INSERT INTO capteurs VALUES (?, ?, ?);", parc_capteurs
)

# Insertion des mesures télémetriques
mesures = [
    ("SOLAR_PANEL_01", "2026-08-30 14:00:00", 1450.75, 230.1),
    ("WIND_TURBINE_02", "2026-08-30 14:00:00", 3400.50, 400.1),
    ("BATTERY_BANK_01", "2026-08-30 14:00:00", -450.00, 48.2),
    ("SOLAR_PANEL_01", "2026-08-30 14:15:00", 1520.00, 230.4),
]
cursor.executemany(
    "INSERT INTO telemetrie_energie (sensor_id, timestamp, puissance_w,"
    " tension_v) VALUES (?, ?, ?, ?);",
    mesures,
)
conn.commit()


# ------------------------------------------------------------------------------
# 🎯 QUÊTE 8 : À VOUS DE JOUER ! (Remplacez les ... par le code exact)
# ------------------------------------------------------------------------------

# ÉTAPE 1 : Extraire l'horodatage, le nom du capteur, son emplacement et sa puissance via un INNER JOIN
sql_exo1 = """
SELECT t.timestamp, t.sensor_id, c.emplacement, t.puissance_w
FROM telemetrie_energie t
... JOIN capteurs c                    -- <--- Indiquer la clause de jonction (INNER)
ON t.sensor_id = c.sensor_code;
"""
cursor.execute(sql_exo1)
telemetrie_enrichie = cursor.fetchall()


# ÉTAPE 2 : Calculez la puissance totale produite par Emplacement (GROUP BY emplacement)
sql_exo2 = """
SELECT c.emplacement, SUM(t.puissance_w) AS puissance_totale
FROM telemetrie_energie t
INNER JOIN capteurs c ... t.sensor_id = c.sensor_code  -- <--- Indiquer le mot-clé de liaison de la jonction (ON)
GROUP BY c.emplacement;
"""
cursor.execute(sql_exo2)
puissance_par_zone = cursor.fetchall()


# ÉTAPE 3 : Extrayez la puissance moyenne uniquement pour le type d'équipement 'Solaire'
sql_exo3 = """
SELECT c.type_equipement, ROUND(AVG(t.puissance_w), 2)
FROM telemetrie_energie t
INNER JOIN capteurs c ON t.sensor_id = c.sensor_code
WHERE c.type_equipement = ...;        -- <--- Indiquer la valeur de filtrage ('Solaire')
"""
cursor.execute(sql_exo3)
stat_solaire = cursor.fetchone()

# Fermeture propre de la connexion
conn.close()


# ------------------------------------------------------------------------------
# 🧪 TESTS DE VALIDATION AUTOMATISÉS
# ------------------------------------------------------------------------------
assert len(telemetrie_enrichie) == 4, (
    f"Erreur Exo 1 : Attendu 4 lignes jointes, trouvé {len(telemetrie_enrichie)}."
)
assert len(telemetrie_enrichie[0]) == 4, (
    "Erreur Exo 1 : La requête doit retourner 4 colonnes (timestamp, sensor_id, emplacement, puissance_w)."
)

assert len(puissance_par_zone) == 3, (
    f"Erreur Exo 2 : Attendu 3 zones d'emplacements, trouvé {len(puissance_par_zone)}."
)

assert stat_solaire[0] == "Solaire" and stat_solaire[1] == 1485.38, (
    f"Erreur Exo 3 : La moyenne du type Solaire doit être 1485.38 W, trouvée {stat_solaire[1]}."
)

print(
    "🏆 Niveau 08 réussi ! Maîtrise des modèles relationnels à 2 tables et des requêtes INNER JOIN."
)

## 🟢 Niveau 09 : Vues SQL (CREATE VIEW) & Joins Externes (LEFT JOIN)

Dans un réseau IoT de production, certains capteurs installés peuvent tomber en panne ou ne jamais avoir émis la moindre trame. Si l'on utilise un `INNER JOIN`, ces capteurs inactifs sont totalement masqués des rapports. 

Pour résoudre cela, on utilise le **`LEFT JOIN`**, combiné aux **Vues SQL (`CREATE VIEW`)** pour simplifier la réutilisation des requêtes complexes.

---

### 💡 `INNER JOIN` vs `LEFT JOIN` vs `CREATE VIEW`

| Concept SQL | Rôle Métier IoT | Syntaxe & Comportement |
| :--- | :--- | :--- |
| **`INNER JOIN`** | Retourne uniquement les capteurs qui possèdent au moins une mesure enregistrée. | Exclut les équipements inactifs ou non raccordés. |
| **`LEFT JOIN`** | Retourne **tous** les capteurs du parc, même ceux sans télémétrie (les valeurs absentes deviennent `NULL`). | Indispensable pour détecter les pannes et capteurs inactifs (`WHERE t.id IS NULL`). |
| **`CREATE VIEW`** | Enregistre une requête SQL complexe sur disque sous la forme d'une **Table Virtuelle**. | Permet d'interroger un rapport complexe avec un simple `SELECT * FROM ma_vue`. |

---

### 🔍 Représentation de l'effet du `LEFT JOIN`

```text
Table [ capteurs ] (3 capteurs)        Table [ telemetrie_energie ] (2 mesures)
┌─────────────────┐                    ┌─────────────────┬─────────────┐
│ SOLAR_PANEL_01  │ ─────────────────> │ SOLAR_PANEL_01  │ 1450.75 W   │
│ WIND_TURBINE_02 │ ─────────────────> │ WIND_TURBINE_02 │ 3400.50 W   │
│ CAPTEUR_HS_03   │ ──(Pas de mesure)─>│ NULL            │ NULL        │
└─────────────────┘                    └─────────────────┴─────────────┘

```

```


In [ ]:
DB_FILE = "iot_energie.db"

# 🔑 ÉTAPE OBLIGATOIRE : Réouvrir la connexion et le curseur
conn = sqlite3.connect(DB_FILE)
cursor = conn.cursor()

cursor.execute("PRAGMA foreign_keys = ON;")

# Nettoyage des anciennes vues et tables
cursor.execute("DROP VIEW IF EXISTS v_synthese_capteurs;")
cursor.execute("DROP TABLE IF EXISTS telemetrie_energie;")
cursor.execute("DROP TABLE IF EXISTS capteurs;")

# 1. Table des capteurs (3 capteurs enregistrés)
cursor.execute("""
CREATE TABLE capteurs (
    sensor_code TEXT PRIMARY KEY,
    emplacement TEXT NOT NULL,
    type_equipement TEXT NOT NULL
);
""")

# 2. Table de télémétrie
cursor.execute("""
CREATE TABLE telemetrie_energie (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    sensor_id TEXT NOT NULL,
    timestamp DATETIME NOT NULL,
    puissance_w REAL NOT NULL,
    tension_v REAL NOT NULL,
    FOREIGN KEY (sensor_id) REFERENCES capteurs(sensor_code)
);
""")

# Insertion des capteurs (Note : CAPTEUR_HS_03 n'a aucune mesure !)
parc_capteurs = [
    ("SOLAR_PANEL_01", "Toiture Bâtiment A", "Solaire"),
    ("WIND_TURBINE_02", "Zone Eolienne Est", "Eolien"),
    ("CAPTEUR_HS_03", "Local Stockage Sud", "Solaire"),
]
cursor.executemany("INSERT INTO capteurs VALUES (?, ?, ?);", parc_capteurs)

# Insertion des mesures (uniquement pour SOLAR_PANEL_01 et WIND_TURBINE_02)
mesures = [
    ("SOLAR_PANEL_01", "2026-08-30 14:00:00", 1450.75, 230.1),
    ("WIND_TURBINE_02", "2026-08-30 14:00:00", 3400.50, 400.1),
]
cursor.executemany(
    "INSERT INTO telemetrie_energie (sensor_id, timestamp, puissance_w,"
    " tension_v) VALUES (?, ?, ?, ?);",
    mesures,
)
conn.commit()


# ------------------------------------------------------------------------------
# 🎯 QUÊTE 9 : À VOUS DE JOUER ! (Remplacez les ... par le code exact)
# ------------------------------------------------------------------------------

# ÉTAPE 1 : Détecter les capteurs inactifs (aucun relevé) en utilisant un LEFT JOIN
sql_exo1 = """
SELECT c.sensor_code, c.emplacement
FROM capteurs c
... JOIN telemetrie_energie t            -- <--- Indiquer le type de jonction externe (LEFT)
ON c.sensor_code = t.sensor_id
WHERE t.id IS NULL;
"""
cursor.execute(sql_exo1)
capteurs_inactifs = cursor.fetchall()


# ÉTAPE 2 : Créer une Vue SQL virtuelle 'v_synthese_capteurs' calculant le total de trames par capteur
sql_exo2 = """
CREATE VIEW IF NOT EXISTS v_synthese_capteurs AS
SELECT 
    c.sensor_code,
    c.type_equipement,
    COUNT(t.id) AS total_trames,
    ROUND(AVG(t.puissance_w), 2) AS puissance_moyenne
FROM capteurs c
LEFT JOIN telemetrie_energie t ON c.sensor_code = t.sensor_id
... BY c.sensor_code;                    -- <--- Indiquer le mot-clé de regroupement (GROUP)
"""
cursor.execute(sql_exo2)
conn.commit()


# ÉTAPE 3 : Interroger simplement la Vue SQL créée comme une table ordinaire
sql_exo3 = """
SELECT sensor_code, total_trames 
FROM v_synthese_capteurs 
WHERE total_trames = ...;                -- <--- Indiquer la valeur pour cibler les capteurs sans trames (0)
"""
cursor.execute(sql_exo3)
rapport_vue = cursor.fetchall()

# Fermeture propre de la connexion
conn.close()


# ------------------------------------------------------------------------------
# 🧪 TESTS DE VALIDATION AUTOMATISÉS
# ------------------------------------------------------------------------------
assert len(capteurs_inactifs) == 1, (
    "Erreur Exo 1 : Attendu 1 capteur inactif décelé via LEFT JOIN, trouvé"
    f" {len(capteurs_inactifs)}."
)
assert capteurs_inactifs[0][0] == "CAPTEUR_HS_03", (
    "Erreur Exo 1 : Le capteur inactif doit être 'CAPTEUR_HS_03', trouvé"
    f" {capteurs_inactifs[0][0]}."
)

assert len(rapport_vue) == 1 and rapport_vue[0][1] == 0, (
    "Erreur Exo 3 : La vue doit retourner 1 capteur ayant un total_trames égal"
    " à 0."
)

print(
    "🏆 Niveau 09 réussi ! Maîtrise des joitures externes LEFT JOIN et création"
    " de Vues SQL (CREATE VIEW)."
)

## 🟢 Niveau 10 : Optimisation, Transactions & Pipeline Analytics (INDEX, ROLLBACK & Pandas)

Félicitations, tu touches au but ! Dans ce dernier niveau, nous passons à l'échelle industrielle en abordant trois compétences fondamentales pour un **Data Engineer IoT** :

1. **L'Indexation (`CREATE INDEX`)** : Accélérer les requêtes sur des millions de séries temporelles.
2. **La Gestion des Transactions (`ROLLBACK`)** : Annuler une opération en cas d'erreur réseau ou matérielle pour garantir l'intégrité de la base.
3. **L'Intégration Data Science (`pd.read_sql_query`)** : Charger directement le résultat SQL dans un DataFrame Pandas pour analyse et visualisation.

---

### 💡 Synthèse des Outils d'Architecture Avancés

| Commande / Outil | Rôle Métier IoT | Syntaxe & Usage |
| :--- | :--- | :--- |
| **`CREATE INDEX`** | Crée un sommaire binaire sur disque pour accélérer les filtres `WHERE` et `ORDER BY` sur le temps. | `CREATE INDEX idx_time ON telemetrie_energie(timestamp)` |
| **`conn.rollback()`** | Annule toutes les modifications non validées depuis le dernier `commit()` en cas de crash. | `except Exception: conn.rollback()` |
| **`pd.read_sql_query()`** | Lit la base SQL et charge le résultat directement dans un DataFrame Pandas. | `df = pd.read_sql_query(sql, conn)` |

---

### 🔍 Pourquoi Indexer la Colonne `timestamp` ?

Sans index, SQLite effectue un **Table Scan** (lecture intégrale de chaque ligne sur disque). Avec des millions de mesures, une requête temporelle prendrait plusieurs secondes. L'index transforme cette recherche en un parcours d'arbre binaire ultrarapide.

```text
Sans Index (Recherche Linéaire O(N))  :  Ligne 1 ──> Ligne 2 ──> ... ──> Ligne 1 000 000 (Lent)
Avec Index (Arbre Binaire O(log N))    :  Index Temporel ──> Pointe directement sur la donnée (Immédiat)

```

```

In [ ]:
DB_FILE = "iot_energie.db"

# 🔑 ÉTAPE OBLIGATOIRE : Réouvrir la connexion
conn = sqlite3.connect(DB_FILE)
cursor = conn.cursor()

cursor.execute("PRAGMA foreign_keys = ON;")

# Re-création propre des tables
cursor.execute("DROP VIEW IF EXISTS v_synthese_capteurs;")
cursor.execute("DROP TABLE IF EXISTS telemetrie_energie;")
cursor.execute("DROP TABLE IF EXISTS capteurs;")

cursor.execute("""
CREATE TABLE capteurs (
    sensor_code TEXT PRIMARY KEY,
    emplacement TEXT NOT NULL,
    type_equipement TEXT NOT NULL
);
""")

cursor.execute("""
CREATE TABLE telemetrie_energie (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    sensor_id TEXT NOT NULL,
    timestamp DATETIME NOT NULL,
    puissance_w REAL NOT NULL,
    tension_v REAL NOT NULL,
    FOREIGN KEY (sensor_id) REFERENCES capteurs(sensor_code)
);
""")

# Remplissage initial
cursor.executemany(
    "INSERT INTO capteurs VALUES (?, ?, ?);",
    [
        ("SOLAR_PANEL_01", "Bâtiment A", "Solaire"),
        ("WIND_TURBINE_02", "Ferme Est", "Eolien"),
    ],
)
cursor.executemany(
    "INSERT INTO telemetrie_energie (sensor_id, timestamp, puissance_w,"
    " tension_v) VALUES (?, ?, ?, ?);",
    [
        ("SOLAR_PANEL_01", "2026-08-30 14:00:00", 1450.75, 230.1),
        ("WIND_TURBINE_02", "2026-08-30 14:00:00", 3400.50, 400.1),
        ("SOLAR_PANEL_01", "2026-08-30 14:15:00", 1520.00, 230.4),
    ],
)
conn.commit()


# ------------------------------------------------------------------------------
# 🎯 QUÊTE 10 : À VOUS DE JOUER ! (Remplacez les ... par le code exact)
# ------------------------------------------------------------------------------

# ÉTAPE 1 : Optimisation — Créez un index nommé 'idx_telemetrie_time' sur la colonne 'timestamp'
sql_exo1 = """
CREATE ... IF NOT EXISTS idx_telemetrie_time  -- <--- Indiquer le mot-clé d'indexation (INDEX)
ON telemetrie_energie(timestamp);
"""
cursor.execute(sql_exo1)
conn.commit()


# ÉTAPE 2 : Sécurité Transactionnelle — Simuler une insertion défectueuse et annuler l'opération avec ROLLBACK
try:
    cursor.execute(
        "INSERT INTO telemetrie_energie (sensor_id, timestamp, puissance_w,"
        " tension_v) VALUES (?, ?, ?, ?)",
        ("CAPTEUR_INEXISTANT", "2026-08-30 15:00:00", 100.0, 230.0),
    )
    conn.commit()
except sqlite3.IntegrityError:
    print("⚠️ Violation de clé étrangère détectée ! Annulation de la transaction.")
    conn....()  # <--- Appeler la méthode d'annulation de transaction (rollback)


# ÉTAPE 3 : Pipeline Data Analytics — Charger le rapport d'énergie dans un DataFrame Pandas via la connexion SQL
sql_exo3 = """
SELECT 
    t.timestamp,
    t.sensor_id,
    c.emplacement,
    t.puissance_w
FROM telemetrie_energie t
INNER JOIN capteurs c ON t.sensor_id = c.sensor_code;
"""

df_analytics = pd.read_sql_query(sql_exo3, ...)  # <--- Transmettre la connexion (conn)


# Fermeture propre de la connexion
conn.close()


# ------------------------------------------------------------------------------
# 🧪 TESTS DE VALIDATION AUTOMATISÉS
# ------------------------------------------------------------------------------
# 1. Vérification de l'existence de l'index
cursor_check = sqlite3.connect(DB_FILE).cursor()
cursor_check.execute(
    "SELECT name FROM sqlite_master WHERE type='index' AND"
    " name='idx_telemetrie_time';"
)
index_verif = cursor_check.fetchone()

assert index_verif is not None, (
    "Erreur Exo 1 : L'index 'idx_telemetrie_time' n'a pas été créé."
)

# 2. Vérification du Rollback (toujours 3 lignes en base)
cursor_check.execute("SELECT COUNT(*) FROM telemetrie_energie;")
nb_lignes = cursor_check.fetchone()[0]
assert nb_lignes == 3, (
    "Erreur Exo 2 : Le rollback n'a pas annulé l'insertion invalide (attendu 3"
    f" lignes, trouvé {nb_lignes})."
)
cursor_check.connection.close()

# 3. Vérification du DataFrame Pandas
assert isinstance(df_analytics, pd.DataFrame), (
    "Erreur Exo 3 : 'df_analytics' doit être un DataFrame Pandas."
)
assert df_analytics.shape == (3, 4), (
    f"Erreur Exo 3 : Dimension attendue (3, 4), trouvée {df_analytics.shape}."
)

print(
    "\n🎉 🏆 NIVEAU 10 ET PROJET GLOBAL RÉUSSIS ! Tu es désormais certifié"
    " 'Architecte Data & Base de Données IoT' !"
)


## 🟢 Niveau 11 : Dashboard Analytics IoT & Visualisation (`utilis.py`)

Pour boucler la chaîne d'ingénierie Data IoT, l'extraction SQL pure laisse place à la restitution graphique. Le module **`utilis.py`** agit comme une couche d'abstraction (Wrapper) qui simplifie le passage entre la base de données SQLite, Pandas et Matplotlib.

---

### 💡 Les Fonctions du Module `utilis.py`

| Fonction | Rôle Métier IoT | Entrée / Sortie |
| :--- | :--- | :--- |
| **`query_to_dataframe()`** | Exécute une requête SQL, gère la connexion/fermeture et convertit automatiquement les timestamps ISO en objets `datetime`. | SQL String $\rightarrow$ `pd.DataFrame` |
| **`plot_telemetrie_temporelle()`** | Affiche les courbes de puissance instantanée ($W$) au cours du temps par capteur. | `df` $\rightarrow$ Graphique Linéaire |
| **`plot_agregation_capteurs()`** | Affiche un diagramme en barres des agrégats (ex: énergie totale par zone). | `df` $\rightarrow$ Graphique Barres |
| **`plot_dashboard_sql_complet()`** | Génère une vue double côte à côte (Flux temporel + Puissance moyenne). | 2 $\times$ `df` $\rightarrow$ Tableau de bord |

---

### 🔍 L'Intérêt de l'Abstraction Python/SQL

En utilisant `query_to_dataframe()`, le code devient plus propre et plus robuste : la gestion du cycle de vie de la connexion (`connect`, `try/finally`, `close`) et le typage temporel de Pandas sont totalement masqués.

```python
# Sans wrapper (Verbeux) :
conn = sqlite3.connect("iot_energie.db")
df = pd.read_sql_query("SELECT * FROM telemetrie_energie", conn)
df["timestamp"] = pd.to_datetime(df["timestamp"])
conn.close()

# Avec utilis.py (1 seule ligne) :
df = query_to_dataframe("iot_energie.db", "SELECT * FROM telemetrie_energie")

```

```

In [ ]:
# ==============================================================================
# 🟢 NIVEAU 11 : INTÉGRATION FINALE ET DASHBOARD ANALYTICS AVEC UTILIS.PY
# ==============================================================================
import os
import sqlite3
import pandas as pd
from utilis import (
    plot_dashboard_sql_complet,
    plot_telemetrie_temporelle,
    query_to_dataframe,
)

DB_FILE = "iot_energie.db"

# 1. Re-création et alimentation du jeu de données final
conn = sqlite3.connect(DB_FILE)
cursor = conn.cursor()
cursor.execute("PRAGMA foreign_keys = ON;")

cursor.execute("DROP TABLE IF EXISTS telemetrie_energie;")
cursor.execute("DROP TABLE IF EXISTS capteurs;")

cursor.execute("""
CREATE TABLE capteurs (
    sensor_code TEXT PRIMARY KEY,
    emplacement TEXT NOT NULL,
    type_equipement TEXT NOT NULL
);
""")

cursor.execute("""
CREATE TABLE telemetrie_energie (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    sensor_id TEXT NOT NULL,
    timestamp DATETIME NOT NULL,
    puissance_w REAL NOT NULL,
    tension_v REAL NOT NULL,
    FOREIGN KEY (sensor_id) REFERENCES capteurs(sensor_code)
);
""")

cursor.executemany(
    "INSERT INTO capteurs VALUES (?, ?, ?);",
    [
        ("SOLAR_PANEL_01", "Toiture Bâtiment A", "Solaire"),
        ("WIND_TURBINE_02", "Ferme Eolienne Est", "Eolien"),
    ],
)

trames_historiques = [
    ("SOLAR_PANEL_01", "2026-08-30 08:00:00", 450.00, 228.5),
    ("WIND_TURBINE_02", "2026-08-30 08:00:00", 2100.00, 400.0),
    ("SOLAR_PANEL_01", "2026-08-30 12:00:00", 1850.50, 231.0),
    ("WIND_TURBINE_02", "2026-08-30 12:00:00", 2500.00, 401.0),
    ("SOLAR_PANEL_01", "2026-08-30 16:00:00", 920.00, 229.5),
    ("WIND_TURBINE_02", "2026-08-30 16:00:00", 3100.00, 399.5),
]
cursor.executemany(
    "INSERT INTO telemetrie_energie (sensor_id, timestamp, puissance_w,"
    " tension_v) VALUES (?, ?, ?, ?);",
    trames_historiques,
)
conn.commit()
conn.close()


# ------------------------------------------------------------------------------
# 🎯 QUÊTE 11 : À VOUS DE JOUER ! (Remplacez les ... par le code exact)
# ------------------------------------------------------------------------------

# ÉTAPE 1 : Extraire la série temporelle complète via le module utilis
sql_series = """
SELECT timestamp, sensor_id, puissance_w 
FROM telemetrie_energie 
ORDER BY timestamp ASC;
"""

df_series = ...(DB_FILE, sql_series)  # <--- Appeler la fonction d'extraction SQL vers DataFrame (query_to_dataframe)


# ÉTAPE 2 : Extraire les agrégats de puissance moyenne par capteur
sql_agreg = """
SELECT 
    sensor_id, 
    ROUND(AVG(puissance_w), 2) AS puissance_moyenne
FROM telemetrie_energie
... BY sensor_id;                     -- <--- Indiquer la clause de regroupement (GROUP)
"""

df_agreg = query_to_dataframe(DB_FILE, sql_agreg)


# ÉTAPE 3 : Générer le Tableau de Bord complet à l'aide de utilis.py
plot_dashboard_sql_complet(
    df_series=df_series,
    df_agreg=...,  # <--- Transmettre le DataFrame d'agrégation (df_agreg)
    title="Tableau de Bord Smart Grid IoT - Analyse Temporelle & Moyennes",
)


# ------------------------------------------------------------------------------
# 🧪 TESTS DE VALIDATION AUTOMATISÉS
# ------------------------------------------------------------------------------
assert isinstance(df_series, pd.DataFrame), (
    "Erreur Exo 1 : df_series doit être un DataFrame Pandas."
)
assert pd.api.types.is_datetime64_any_dtype(df_series["timestamp"]), (
    "Erreur Exo 1 : La colonne timestamp doit être convertie en datetime par"
    " utilis.py."
)

assert "puissance_moyenne" in df_agreg.columns, (
    "Erreur Exo 2 : La colonne 'puissance_moyenne' doit figurer dans df_agreg."
)
assert len(df_agreg) == 2, (
    f"Erreur Exo 2 : Attendu 2 capteurs agrégés, trouvé {len(df_agreg)}."
)

print(
    "🏆 Niveau 11 réussi ! Le pipeline complet (SQL -> Pandas -> Dashboard"
    " utilis.py) est parfaitement fonctionnel !"
)